<a href="https://colab.research.google.com/github/Kajlid/ft-lora/blob/axels-branch/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install dependencies

In [1]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps \
    git+https://github.com/unslothai/unsloth.git@nightly \
    git+https://github.com/unslothai/unsloth-zoo.git

# 👇 Single, consistent install of huggingface_hub that matches Unsloth's requirement
!pip install --upgrade "huggingface_hub>=0.34.0,<1.0.0"


In [2]:
# Login to HuggingFace

#from huggingface_hub import notebook_login
#notebook_login()

from huggingface_hub import HfApi

# 🔐 TEMP: paste your personal HF token here.
# Make sure you NEVER commit this notebook with the token included.
HF_TOKEN = "hf_..."

# Create an API client that always uses this token
api = HfApi(token=HF_TOKEN)

# Optional sanity check – if this works, auth is fine
print(api.whoami())


{'type': 'user', 'id': '692d684dfa038bf49fb7c0b0', 'name': 'AxelHolst', 'fullname': 'Axel Barck-Holst', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1767225600, 'isPro': False, 'avatarUrl': '/avatars/7f2267f6eee07c05480d29c13af4c380.svg', 'orgs': [{'type': 'org', 'id': '69298f933aebd54408a5e33e', 'name': 'ft-lora', 'fullname': 'ID2223 Legendariskt bästa gruppen', 'email': 'kajsa.lidin@gmail.com', 'canPay': False, 'billingMode': 'postpaid', 'periodEnd': None, 'avatarUrl': 'https://www.gravatar.com/avatar/7d0331cec95d511ed2d37de87799c08e?d=retro&size=100', 'roleInOrg': 'admin', 'isEnterprise': False}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'ID2223FDLORAToken', 'role': 'fineGrained', 'createdAt': '2025-12-02T10:30:59.698Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write', 'post.write'], 'scoped': [{'entity': {'_id': '6929ab00b478921448035583', 'type': 'model', 'name': 'ft-lora/llama3.2-3b-instruct-finetuned'}, 'permissions': ['

In [3]:
#from huggingface_hub import HfApi

# Make sure HuggingFace repo exists inside the 'ft-lora' organization
api = HfApi()
api.create_repo(
    repo_id="ft-lora/llama3.2-1b-instruct-finetuned",  # NEW NAME
    repo_type="model",
    exist_ok=True,
    token=HF_TOKEN,  # extra explicit, just in case
)

RepoUrl('https://huggingface.co/ft-lora/llama3.2-1b-instruct-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='ft-lora/llama3.2-1b-instruct-finetuned')

## Chosen Foundation model

In [4]:
from unsloth import FastLanguageModel
import torch

# --- PATCH huggingface_hub constant for Unsloth ---
import huggingface_hub
from huggingface_hub import constants as hf_constants

# If the constant doesn't exist, define it (False = don't use hf_transfer)
if not hasattr(hf_constants, "HF_HUB_ENABLE_HF_TRANSFER"):
    hf_constants.HF_HUB_ENABLE_HF_TRANSFER = False
# --- end patch ---


# max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
max_seq_length = 1024
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct", # smaller model: "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.11.6 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


## Data preparation

In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",     # 3.2?
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

We now use standardize_sharegpt to convert ShareGPT style datasets into HuggingFace's generic format. This changes the dataset from looking like:

```
{"from": "system", "value": "You are an assistant"}
{"from": "human", "value": "What is 2+2?"}
{"from": "gpt", "value": "It's 4."}
```


to
```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [7]:
from unsloth.chat_templates import standardize_sharegpt
dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched = True,)
dataset = dataset.shuffle(seed=3407).select(range(20000)) #For the smaller model, it’s cheap to see more data.

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Look at conversation 5

In [8]:
dataset[5]["conversations"]

[{'content': 'Explain how the principle of non-contradiction can be applied to prove that any prime number greater than 2 cannot be expressed as the sum of two smaller prime numbers, and use this principle to demonstrate the truth or falsity of the given statement.',
  'role': 'user'},
 {'content': "The principle of non-contradiction states that a statement and its negation cannot both be true at the same time. \nTo prove that any prime number greater than 2 cannot be expressed as the sum of two smaller prime numbers, we can assume the opposite and show that it leads to a contradiction. \nLet's assume that there exists a prime number p greater than 2 that can be expressed as the sum of two smaller prime numbers, say p = q + r, where q and r are smaller prime numbers. \nSince q and r are smaller than p, they cannot be equal to p. Therefore, they must be less than p. \nBut since p is a prime number, it can only be divided by 1 and itself. Therefore, neither q nor r can divide p. \nNow, l

And we see how the chat template transformed these conversations.

[Notice] Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

## Training the model

Now let's use Huggingface TRL's SFTTrainer! More docs here: TRL SFT docs. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    # packing = False,
    packing = True,             # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Set this for 1 full training run.
        #max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1, # Logs training progress
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc

        # Checkpointing model
        save_strategy="steps",       # Save checkpoints during training
        save_steps=10,               # Save every 10 steps
        save_total_limit=3,         # Keep only the last 3 checkpoints

        # The HuggingFace organization to push to
        push_to_hub=True,     # set to True for continuous saving in HF
        hub_model_id="ft-lora/llama3.2-1b-instruct-finetuned",  # repo name
        hub_strategy="checkpoint",   # uploads checkpoints during training
        hub_token = HF_TOKEN,
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/20000 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [10]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/20000 [00:00<?, ? examples/s]

We verify masking is actually done:

In [11]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

"<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nExplain how the principle of non-contradiction can be applied to prove that any prime number greater than 2 cannot be expressed as the sum of two smaller prime numbers, and use this principle to demonstrate the truth or falsity of the given statement.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe principle of non-contradiction states that a statement and its negation cannot both be true at the same time. \nTo prove that any prime number greater than 2 cannot be expressed as the sum of two smaller prime numbers, we can assume the opposite and show that it leads to a contradiction. \nLet's assume that there exists a prime number p greater than 2 that can be expressed as the sum of two smaller prime numbers, say p = q + r, where q and r are smaller prime numbers. \nSince q a

In [12]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

"                                                                                        The principle of non-contradiction states that a statement and its negation cannot both be true at the same time. \nTo prove that any prime number greater than 2 cannot be expressed as the sum of two smaller prime numbers, we can assume the opposite and show that it leads to a contradiction. \nLet's assume that there exists a prime number p greater than 2 that can be expressed as the sum of two smaller prime numbers, say p = q + r, where q and r are smaller prime numbers. \nSince q and r are smaller than p, they cannot be equal to p. Therefore, they must be less than p. \nBut since p is a prime number, it can only be divided by 1 and itself. Therefore, neither q nor r can divide p. \nNow, let's consider the number (p-2). Since p is greater than 2, we know that (p-2) is a positive integer. \nWe can express (p-2) as the sum of q and r: (p-2) = q + r. \nAdding 2 to both sides, we get: p = (q+2) + (r+2

We can see the System and Instruction prompts are successfully masked!

### Show current memory stats

In [13]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
1.203 GB of memory reserved.


In [14]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,000 | Num Epochs = 1 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.235700
2,0.931900
3,1.008900
4,0.925200
5,1.064700
6,1.109300
7,0.878700
8,1.030800
9,1.003400
10,1.089100


In [15]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

6637.1914 seconds used for training.
110.62 minutes used for training.
Peak reserved memory = 2.355 GB.
Peak reserved memory for training = 1.152 GB.
Peak reserved memory % of max memory = 15.976 %.
Peak reserved memory for training % of max memory = 7.815 %.


## Inference

Let's run the model! You can change the instruction and input - leave the output blank!

[NEW] Try 2x faster inference in a free Colab for Llama-3.1 8b Instruct here

We use min_p = 0.1 and temperature = 1.5.

In [16]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 1.5, min_p = 0.1)
tokenizer.batch_decode(outputs)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContinue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe next number in the sequence should be 13.\n\nThe Fibonacci sequence is a series of numbers in which each number is the sum of the two preceding numbers. This means that a number before a number is added to get a number after it, except when there is only one preceding number, in which case it is added']

You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [17]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 6765, 10946, 17715, 28657, 46368, 75025, 121393, 198612, 325147, 526702, 843873, 1344130, 2178917, 3495706, 5772516, 9666720,


### Saving, loading finetuned models

To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

[NOTE] This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [18]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.json')

In [19]:
# Save the finetuned model to HuggingFace
# Organization name: ft-lora

# Online saving
# model.push_to_hub("ft-lora/llama3.2-3b-instruct-finetuned")
# tokenizer.push_to_hub("ft-lora/llama3.2-3b-instruct-finetuned")


Now if you want to load the LoRA adapters we just saved for inference, set False to True:

In [20]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # the model used for training
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
The tall tower in the capital of France is a prominent feature in the city. It stands at an impressive height and dominates the landscape. The tower is built into a slope, with the highest point being at the edge of the hill where it can reach. This unique location is a result of the tower's original purpose as a water supply storage system. The water tower now serves as a modern office building with numerous windows and a bright atrium. As people enter and exit the building, their footsteps echo through the halls, making the office spac

### Saving to float16 for VLLM

We also support saving to float16 directly. Select merged_16bit for float16 or merged_4bit for int4. We also allow lora adapters as a fallback. Use push_to_hub_merged to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [23]:
# Merge to 16bit, standard HuggingFace model
# LoRA weights are merged into the base model
# Needed for GGUF
#if True: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
#if True: model.push_to_hub_merged("ft-lora/llama3.2-3b-instruct-finetuned", tokenizer, save_method = "merged_16bit")

if True:
    model.save_pretrained_merged(
        "model_1b",                # NEW LOCAL DIR
        tokenizer,
        save_method="merged_16bit",
    )

if True:
    model.push_to_hub_merged(
        "ft-lora/llama3.2-1b-instruct-finetuned",  # HF repo id
        tokenizer,
        save_method="merged_16bit",
        token=HF_TOKEN,
    )



# Merge to 4bit, better for for CPU inference
# if True: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit_forced",)
# if True: model.push_to_hub_merged("ft-lora/llama3.2-3b-instruct-finetuned", tokenizer, save_method = "merged_4bit_forced")

# Just LoRA adapters
# if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
# if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 8886.24it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:09<00:00, 129.94s/it]


Unsloth: Merge process complete. Saved to `/content/model_1b`


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...netuned/model.safetensors:   1%|1         | 25.2MB / 2.47GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:37<00:00, 97.87s/it]


Unsloth: Merge process complete. Saved to `/content/ft-lora/llama3.2-1b-instruct-finetuned`


## GGUF / llama.cpp Conversion

In [24]:
!git clone https://github.com/ggml-org/llama.cpp.git
%cd llama.cpp

!pip install -r requirements.txt
!pip install -U transformers huggingface_hub


Cloning into 'llama.cpp'...
remote: Enumerating objects: 70157, done.
remote: Counting objects: 100% (337/337), done.
remote: Compressing objects: 100% (227/227), done.
remote: Total 70157 (delta 222), reused 110 (delta 110), pack-reused 69820 (from 3)
Receiving objects: 100% (70157/70157), 215.43 MiB | 30.51 MiB/s, done.
Resolving deltas: 100% (50706/50706), done.
/content/llama.cpp
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.9 MB/s eta 0:00:00
    

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 138.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalling transformers-4.57.2:
      Successfully uninstalled transformers-4.57.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2025.11.6 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,<=4.57.2,>=4.51.3, but you have transformers 4.57.3 which is incompatible.


Switch to CPU here?

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp.git
%cd llama.cpp

In [ ]:
!pip install -r requirements.txt
!pip install -U transformers huggingface_hub

In [ ]:
%cd /content/llama.cpp

In [ ]:
!ls

In [ ]:
ls -lh /content/llama.cpp/ft-lora/llama3.2-3b-instruct-finetuned


In [26]:
# Define directory to save merged model
# merged_model_dir = "/content/lora_model_merged"

# Save merged model + tokenizer
# model.save_pretrained(merged_model_dir, tokenizer)
# merged_model.save_pretrained("/content/merged_model", tokenizer)

# Convert to GGUF using llama.cpp script
#!python convert_hf_to_gguf.py /content/llama.cpp/ft-lora/llama3.2-3b-instruct-finetuned \
    #--outfile /content/llama3.2-3b-instruct-finetuned.gguf \
    #--outtype auto
"""
!python convert_hf_to_gguf.py /content/llama.cpp/model \
    --outfile /content/llama3.2-3b-instruct-finetuned.gguf \
    --outtype auto
"""
%cd /content/llama.cpp

!python convert_hf_to_gguf.py /content/model_1b \
    --outfile /content/llama3.2-1b-instruct-finetuned.gguf \
    --outtype auto


#!python convert_hf_to_gguf.py /content/lora_model \
    #--outfile /content/llama3.2-3b-instruct-finetuned.gguf \
    #--outtype auto

/content/llama.cpp
INFO:hf-to-gguf:Loading model: model_1b
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:hf-to-gguf:choosing --outtype bf16 from first tensor type (torch.bfloat16)
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {32}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> BF16, shape = {2048, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> BF16, shape = {8192, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfloat16 --> BF16, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.bfloat16 --> BF16, shape = {2048, 8192}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.bfloat16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_

In [27]:
!ls -lh /content/llama3.2-1b-instruct-finetuned.gguf


-rw-r--r-- 1 root root 2.4G Dec  2 16:24 /content/llama3.2-1b-instruct-finetuned.gguf


In [28]:
from huggingface_hub import HfApi

HF_TOKEN = HF_TOKEN  # reuse the one you already defined earlier

api = HfApi(token=HF_TOKEN)
repo_id = "ft-lora/llama3.2-1b-gguf-auto"  # name it as you like

# Create (or reuse) the repo
api.create_repo(
    repo_id=repo_id,
    repo_type="model",
    exist_ok=True,
    token=HF_TOKEN,
)

# Upload the GGUF file
api.upload_file(
    path_or_fileobj="/content/llama3.2-1b-instruct-finetuned.gguf",
    path_in_repo="llama3.2-1b-instruct-finetuned.gguf",
    repo_id=repo_id,
    token=HF_TOKEN,
)


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b-instruct-finetuned.gguf:   1%|1         | 25.6MB / 2.48GB            

CommitInfo(commit_url='https://huggingface.co/ft-lora/llama3.2-1b-gguf-auto/commit/1d611529339de3f7ff423e5acf56a8cedf0a3fd7', commit_message='Upload llama3.2-1b-instruct-finetuned.gguf with huggingface_hub', commit_description='', oid='1d611529339de3f7ff423e5acf56a8cedf0a3fd7', pr_url=None, repo_url=RepoUrl('https://huggingface.co/ft-lora/llama3.2-1b-gguf-auto', endpoint='https://huggingface.co', repo_type='model', repo_id='ft-lora/llama3.2-1b-gguf-auto'), pr_revision=None, pr_num=None)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "ft-lora/llama3.2-1b-gguf-q4km"  # NEW REPO
api.create_repo(repo_id, repo_type="model", exist_ok=True)

api.upload_file(
    path_or_fileobj="/content/llama3.2-1b-instruct-finetuned.gguf",
    path_in_repo="llama3.2-1b-instruct-finetuned.gguf",
    repo_id=repo_id,
)